# Week 6 · Day 5 — Checkpoint 6, Part A: intervals and tests on Nur data (50 pts)

Five tasks. For each: code cell(s) with the result printed, then a Markdown cell with **two sentences of interpretation** a risk manager can read.
Parts B and C (the test-choice quiz and the interview questions) go into `quiz.md`, not here.
Submit this notebook **executed** (Kernel → Restart & Run All, then save) to `submissions/week-06/day-5/`.

In [1]:
from pathlib import Path
import hashlib
import sqlite3
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
DATA = Path("../..")           # this notebook lives in data/python/week6/, so "../.." is the data/ folder
AS_OF = pd.Timestamp("2026-10-01")
con = sqlite3.connect(DATA / "raw" / "nur.db")
rng = np.random.default_rng(2026)   # one seeded generator: the same "random" numbers on every run


def check(value, fingerprints, dp=0):
    """Right or wrong without showing the answer: the rounded value's fingerprint must be one of the expected ones."""
    fp = hashlib.sha256(f"nur-w6:{round(float(value), dp):.{dp}f}".encode()).hexdigest()[:12]
    assert fp in fingerprints, "not yet: this result is not the expected one — check your function and run again"

In [2]:
# The three frames of the week (built exactly as in week 5; run, do not edit)
loans = pd.read_sql("""
    SELECT l.loan_id, l.disbursed_at, l.principal, l.term_months, l.risk_grade, l.status,
           p.code, p.name AS product, a.channel, a.credit_score, b.branch_name
    FROM loans l JOIN products p USING (product_id) JOIN applications a USING (application_id)
    JOIN branches b ON b.branch_id = l.branch_id""", con, parse_dates=["disbursed_at"])
inst = pd.read_sql("SELECT * FROM installments", con, parse_dates=["due_date"])
pay = (pd.read_sql("SELECT * FROM payments", con, parse_dates=["paid_at"])
         .sort_values("payment_id").drop_duplicates(["installment_id", "paid_at", "amount"]))   # 52 850 clean payments

# f: one row per loan whose FIRST instalment fell due on or before as-of - 30 days; fpd30 = 1 if not fully paid within 30 days
first = inst[(inst["installment_no"] == 1) & (inst["due_date"] <= AS_OF - pd.Timedelta(days=30))]
paid1 = pay[pay["paid_at"] <= AS_OF].groupby("installment_id", as_index=False).agg(paid=("amount", "sum"), first_paid=("paid_at", "min"))
f = first.merge(paid1, on="installment_id", how="left").fillna({"paid": 0})
f["fpd30"] = ((f["paid"] < f["total_due"]) | ((f["first_paid"] - f["due_date"]).dt.days > 30)).astype(int)
f = f.merge(loans, on="loan_id", validate="1:1")

# x: one row per clean payment, with its due date, days late and the loan's grade / product
x = pay.merge(inst[["installment_id", "due_date"]], on="installment_id").merge(
        loans[["loan_id", "risk_grade", "code", "channel"]], on="loan_id")
x["days_late"] = (x["paid_at"].dt.normalize() - x["due_date"]).dt.days
x["late"] = (x["paid_at"] > x["due_date"]).astype(int)
assert len(loans) == 7828 and len(f) == 7099 and f["fpd30"].sum() == 91 and len(x) == 52850
print(f"loans {len(loans)} · FPD30 {f.fpd30.sum()} / {len(f)} = {f.fpd30.mean():.2%} · clean payments {len(x)}")
apps = pd.read_sql("SELECT application_id, channel, applied_at, status FROM applications", con, parse_dates=["applied_at"])
from statsmodels.stats.proportion import proportion_confint, proportions_ztest, confint_proportions_2indep, proportion_effectsize
from statsmodels.stats.power import NormalIndPower, zt_ind_solve_power

loans 7828 · FPD30 91 / 7099 = 1.28% · clean payments 52850


## A1. CI for the average principal by product (t interval) — 10 pts

For each of the five products: n, mean, sd, the 95 % t interval, the half-width as % of the mean. Then: which product's average do you know least precisely, and why (n or sd)?

In [3]:
# your code

**Interpretation (two sentences):** …

## A2. CI for the approval rate by channel (Wilson) — 10 pts

Approved = status starting with `approved`; denominator = **decided** applications (cancelled out). Wilson 95 % per channel. Do the app and branch intervals overlap?

In [4]:
# your code

**Interpretation (two sentences):** …

## A3. Test: late-payment rate, customer segment sme vs mass — 10 pts

Late = paid after the due date, on clean payments (52 850); the segment is the customer's (`customers.segment`, through `loans.customer_id`). Two-proportion z-test or chi-square on the 2 × 2. Report the p-value, the difference in points and the relative risk; one sentence on what the grain (payments, not loans) does to the test, and the loan-level check.

In [5]:
# your code

**Interpretation (two sentences):** …

## A4. Test: days late, Consumer loans of grade B vs grade C — 10 pts

Mann–Whitney U (two-sided). Before the test: the shape (median, mean, skew, share paid late) and one sentence on why not a t-test. After: the p-value and the probability of superiority U / (n1·n2).

In [6]:
# your code

**Interpretation (two sentences):** …

## A5. Power: loans needed to detect +2 pp on the app approval rate — 10 pts

Baseline = the app's decided approval rate (A2). Cohen's h, alpha 0.05 two-sided, power 0.80. Then: how many months of app applications is that, at the 2026 monthly volume of decided app applications (January–September, whole months)?

In [7]:
# your code

**Interpretation (two sentences):** …